# EB-JEPA Prescribed Axes v3

**Goal:** reproduce EB-JEPA (LeCun et al., 2602.03604) and compare against prescribed axes.

**Checkpointing:** every 200 batches (~30 min) and after each epoch → Drive.
**Resume:** from the exact batch on reconnect.


In [ ]:
# CELL 1: Setup
!pip install -q einops fire omegaconf ruamel.yaml pymunk imageio seaborn

import os, json, torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive: {DRIVE_BASE}')
print(f'Existing: {os.listdir(DRIVE_BASE) or "empty"}')

In [ ]:
# CELL 2: Extract eb_jepa code
import zipfile, os

EB_DIR = '/content/eb_jepa'
if os.path.exists(os.path.join(EB_DIR, 'eb_jepa', 'jepa.py')):
    print('Already extracted')
else:
    from google.colab import files
    print('Upload eb_jepa_colab.zip:')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall(EB_DIR)
    print('Extracted')

os.chdir(EB_DIR)
!pip install -e . -q
from eb_jepa.jepa import JEPA
print('OK')

In [ ]:
# CELL 3: Write run_experiment_v3.py
import os
script_dir = '/content/eb_jepa/experiments/prescribed_axes'
os.makedirs(script_dir, exist_ok=True)
script_path = os.path.join(script_dir, 'run_experiment_v3.py')

SCRIPT = '"""\nEB-JEPA Prescribed Axes Experiment v3.\n\nReproduces LeCun et al. (2602.03604) Two Rooms experiment with prescribed axes.\nClean implementation: no patches, no monkey-patching.\n\nPer-epoch: checkpoint + results.json + planning eval saved to Google Drive.\nResume: automatic from Drive checkpoint.\n"""\n\nimport copy\nimport json\nimport os\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport yaml\nfrom omegaconf import OmegaConf\nfrom torch.amp import GradScaler, autocast\nfrom torch.optim import AdamW\nfrom tqdm import tqdm\n\n# EB-JEPA imports\nfrom eb_jepa.architectures import (\n    ImpalaEncoder, InverseDynamicsModel, Projector, RNNPredictor,\n)\nfrom eb_jepa.datasets.utils import init_data\nfrom eb_jepa.jepa import JEPA, JEPAProbe\nfrom eb_jepa.logging import get_logger\nfrom eb_jepa.losses import SquareLossSeq, VC_IDM_Sim_Regularizer\nfrom eb_jepa.schedulers import CosineWithWarmup\nfrom eb_jepa.state_decoder import MLPXYHead\nfrom eb_jepa.training_utils import (\n    load_config, save_checkpoint, setup_device, setup_seed,\n)\nfrom eb_jepa.planning import main_eval\n\nlogger = get_logger(__name__)\n\nDEVICE = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n\n\n# ================================================================\n# Prescribed and Hybrid encoders\n# ================================================================\n\nclass PrescribedEncoder(nn.Module):\n    """Uses agent coordinates (x,y) instead of pixels. Output: [B, D, T, 1, 1]."""\n\n    def __init__(self, prescribed_dim=2, output_dim=512, hidden_dim=256, final_ln=True):\n        super().__init__()\n        self.mlp_output_dim = output_dim\n        self.prescribed_dim = prescribed_dim\n        self.projection = nn.Sequential(\n            nn.Linear(prescribed_dim, hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, output_dim),\n        )\n        self.final_ln = nn.LayerNorm(output_dim) if final_ln else nn.Identity()\n\n    def forward(self, observations, locations=None):\n        if locations is None:\n            raise ValueError("PrescribedEncoder requires locations")\n        loc = locations.permute(0, 2, 1)  # [B, 2, T] -> [B, T, 2]\n        B, T, _ = loc.shape\n        features = self.projection(loc.reshape(B * T, -1))\n        features = self.final_ln(features)\n        features = features.reshape(B, T, -1)\n        return features.transpose(1, 2).unsqueeze(-1).unsqueeze(-1)\n\n\nclass HybridEncoder(nn.Module):\n    """Combines prescribed coordinates with free pixel features. Output: [B, D, T, 1, 1]."""\n\n    def __init__(self, pixel_encoder, prescribed_dim=2, prescribed_output_dim=128, final_ln=True):\n        super().__init__()\n        self.pixel_encoder = pixel_encoder\n        self.prescribed_dim = prescribed_dim\n        total_dim = pixel_encoder.mlp_output_dim\n        free_dim = total_dim - prescribed_output_dim\n        self.prescribed_output_dim = prescribed_output_dim\n        self.free_dim = free_dim\n        self.mlp_output_dim = total_dim\n        self.prescribed_projection = nn.Sequential(\n            nn.Linear(prescribed_dim, prescribed_output_dim),\n            nn.ReLU(),\n            nn.Linear(prescribed_output_dim, prescribed_output_dim),\n        )\n        self.pixel_reduction = nn.Linear(pixel_encoder.mlp_output_dim, free_dim)\n        self.final_ln = nn.LayerNorm(total_dim) if final_ln else nn.Identity()\n\n    def forward(self, observations, locations=None):\n        if locations is None:\n            raise ValueError("HybridEncoder requires locations")\n        pixel_features = self.pixel_encoder(observations)\n        B, D, T, _, _ = pixel_features.shape\n        pixel_feat = pixel_features.squeeze(-1).squeeze(-1).transpose(1, 2)\n        pixel_feat = self.pixel_reduction(pixel_feat)\n        loc = locations.permute(0, 2, 1)\n        loc_flat = loc.reshape(B * T, -1)\n        prescribed_feat = self.prescribed_projection(loc_flat).reshape(B, T, -1)\n        combined = torch.cat([prescribed_feat, pixel_feat], dim=-1)\n        combined = self.final_ln(combined)\n        return combined.transpose(1, 2).unsqueeze(-1).unsqueeze(-1)\n\n\n# ================================================================\n# PrescribedJEPA — passes locations to encoder\n# ================================================================\n\nclass PrescribedJEPA(JEPA):\n    """JEPA that passes locations to encoder during training and planning."""\n\n    def __init__(self, encoder, aencoder, predictor, regularizer, predcost):\n        super().__init__(encoder, aencoder, predictor, regularizer, predcost)\n        self._current_locations = None\n\n    def set_locations_for_planning(self, loc):\n        self._current_locations = loc\n\n    def clear_planning_locations(self):\n        self._current_locations = None\n\n    @torch.no_grad()\n    def encode(self, observations):\n        if self._current_locations is not None and hasattr(self.encoder, \'prescribed_dim\'):\n            return self.encoder(observations, locations=self._current_locations)\n        if hasattr(self.encoder, \'prescribed_dim\'):\n            raise ValueError("PrescribedEncoder.encode() needs locations. Call set_locations_for_planning().")\n        return self.encoder(observations)\n\n    def unroll(self, observations, actions, nsteps=1, unroll_mode="parallel",\n               ctxt_window_time=1, compute_loss=True, return_all_steps=False,\n               locations=None):\n        if locations is not None and hasattr(self.encoder, \'prescribed_dim\'):\n            state = self.encoder(observations, locations=locations)\n        elif self._current_locations is not None and hasattr(self.encoder, \'prescribed_dim\'):\n            state = self.encoder(observations, locations=self._current_locations)\n        else:\n            state = self.encoder(observations)\n\n        context_length = getattr(self.predictor, "context_length", 0)\n\n        if compute_loss:\n            rloss, rloss_unweight, rloss_dict = self.regularizer(state, actions)\n            ploss = 0.0\n        else:\n            rloss = rloss_unweight = rloss_dict = ploss = None\n\n        actions_encoded = self.action_encoder(actions) if actions is not None else None\n        all_steps = [] if return_all_steps else None\n\n        if unroll_mode == "parallel":\n            predicted_states = state\n            for _ in range(nsteps):\n                predicted_states = self.predictor(predicted_states, actions_encoded)[:, :, :-1]\n                if return_all_steps:\n                    all_steps.append(predicted_states)\n                predicted_states = torch.cat(\n                    (state[:, :, :context_length], predicted_states), dim=2\n                )\n                if compute_loss:\n                    ploss += self.predcost(state, predicted_states) / nsteps\n\n        elif unroll_mode == "autoregressive":\n            if actions is not None and nsteps > actions.size(2):\n                raise ValueError(f"nsteps ({nsteps}) > actions ({actions.size(2)})")\n            effective_ctxt_window = 1 if self.single_unroll else ctxt_window_time\n            predicted_states = state[:, :, :effective_ctxt_window]\n            for i in range(nsteps):\n                context_states = predicted_states[:, :, -effective_ctxt_window:]\n                if actions_encoded is not None:\n                    context_actions = actions_encoded[\n                        :, :, max(0, i + 1 - effective_ctxt_window): i + 1\n                    ]\n                else:\n                    context_actions = None\n                pred_step = self.predictor(context_states, context_actions)[:, :, -1:]\n                predicted_states = torch.cat([predicted_states, pred_step], dim=2)\n                if return_all_steps:\n                    all_steps.append(predicted_states.clone())\n                if compute_loss:\n                    ploss += torch.nn.functional.mse_loss(\n                        pred_step, state[:, :, i + 1: i + 2]\n                    ) / nsteps\n        else:\n            raise ValueError(f"Unknown unroll_mode: {unroll_mode}")\n\n        if compute_loss:\n            losses = (ploss + rloss, rloss, rloss_unweight, rloss_dict, ploss)\n        else:\n            losses = None\n\n        return (all_steps if return_all_steps else predicted_states), losses\n\n\n# ================================================================\n# Condition configs\n# ================================================================\n\nCONDITIONS = {\n    \'free\': {},\n    \'prescribed\': {\'encoder_type\': \'prescribed\'},\n    \'hybrid\': {\'encoder_type\': \'hybrid\'},\n    \'prescribed_no_idm\': {\'encoder_type\': \'prescribed\', \'idm_coeff\': 0},\n    \'prescribed_no_vicreg\': {\'encoder_type\': \'prescribed\', \'std_coeff\': 0, \'cov_coeff\': 0},\n    \'prescribed_no_sim\': {\'encoder_type\': \'prescribed\', \'sim_coeff_t\': 0},\n}\n\n\n# ================================================================\n# Drive helpers\n# ================================================================\n\ndef _drive_dir(mode, drive_base):\n    d = os.path.join(drive_base, mode)\n    os.makedirs(d, exist_ok=True)\n    return d\n\n\ndef _save_to_drive(mode, drive_base, local_folder, results_log, epoch_stats):\n    dst = _drive_dir(mode, drive_base)\n    src_ckpt = os.path.join(local_folder, "latest.pth.tar")\n    if os.path.exists(src_ckpt):\n        shutil.copy2(src_ckpt, os.path.join(dst, "latest.pth.tar"))\n    with open(os.path.join(dst, "results.json"), "w") as f:\n        json.dump(results_log, f, indent=2)\n    stats_path = os.path.join(dst, "encoder_stats.json")\n    existing = []\n    if os.path.exists(stats_path):\n        with open(stats_path) as f:\n            existing = json.load(f)\n    existing.append(epoch_stats)\n    with open(stats_path, "w") as f:\n        json.dump(existing, f, indent=2)\n\n\ndef _load_resume(mode, drive_base, device, jepa, jepa_opt, jepa_sched, xy_head,\n                 probe_opt=None, probe_sched=None):\n    ckpt_path = os.path.join(_drive_dir(mode, drive_base), "latest.pth.tar")\n    if not os.path.exists(ckpt_path):\n        return 0, 0, []\n    logger.info(f"Resuming from: {ckpt_path}")\n    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)\n    jepa.load_state_dict(ckpt["model_state_dict"])\n    jepa_opt.load_state_dict(ckpt["optimizer_state_dict"])\n    jepa_sched.load_state_dict(ckpt["scheduler_state_dict"])\n    if "xy_head_state_dict" in ckpt:\n        xy_head.load_state_dict(ckpt["xy_head_state_dict"])\n    if probe_opt and "probe_optimizer_state_dict" in ckpt:\n        probe_opt.load_state_dict(ckpt["probe_optimizer_state_dict"])\n    if probe_sched and "probe_scheduler_state_dict" in ckpt:\n        probe_sched.load_state_dict(ckpt["probe_scheduler_state_dict"])\n    saved_epoch = ckpt["epoch"]\n    saved_step = ckpt.get("batch_idx", -1)  # -1 means end of epoch\n    if saved_step == -1:\n        # Full epoch completed — start next epoch from batch 0\n        start_epoch = saved_epoch + 1\n        start_step = 0\n    else:\n        # Mid-epoch checkpoint — resume same epoch from next batch\n        start_epoch = saved_epoch\n        start_step = saved_step + 1\n    del ckpt\n    torch.cuda.empty_cache()\n    results_path = os.path.join(_drive_dir(mode, drive_base), "results.json")\n    results_log = []\n    if os.path.exists(results_path):\n        with open(results_path) as f:\n            results_log = json.load(f)\n    logger.info(f"Resumed: epoch {start_epoch}, batch {start_step}, {len(results_log)} results loaded")\n    return start_epoch, start_step, results_log\n\n\n# ================================================================\n# Encoder stats for drift analysis\n# ================================================================\n\n@torch.no_grad()\ndef compute_encoder_stats(jepa, loader, device, locations_available):\n    all_features = []\n    n_batches = min(10, len(loader))\n    for i, (x, a, loc, _, _) in enumerate(loader):\n        if i >= n_batches:\n            break\n        x, loc = x.to(device), loc.to(device)\n        if locations_available and hasattr(jepa.encoder, \'prescribed_dim\'):\n            feat = jepa.encoder(x, locations=loc)\n        else:\n            feat = jepa.encoder(x)\n        B, D, T, _, _ = feat.shape\n        feat_flat = feat.squeeze(-1).squeeze(-1).permute(0, 2, 1).reshape(-1, D)\n        all_features.append(feat_flat.cpu())\n    all_features = torch.cat(all_features, dim=0)\n    return {\n        \'mean_per_dim\': all_features.mean(dim=0).tolist()[:20],\n        \'std_per_dim\': all_features.std(dim=0).tolist()[:20],\n        \'global_mean\': all_features.mean().item(),\n        \'global_std\': all_features.std().item(),\n        \'n_samples\': all_features.shape[0],\n    }\n\n\n# ================================================================\n# Build encoder\n# ================================================================\n\ndef build_encoder(encoder_type, cfg, data_config):\n    if encoder_type == \'free\':\n        return ImpalaEncoder(\n            width=1,\n            stack_sizes=(16, cfg.model.henc, cfg.model.dstc),\n            num_blocks=2, dropout_rate=None, layer_norm=False,\n            input_channels=cfg.model.dobs, final_ln=True,\n            mlp_output_dim=512,\n            input_shape=(cfg.model.dobs, data_config.img_size, data_config.img_size),\n        )\n    elif encoder_type == \'prescribed\':\n        return PrescribedEncoder(prescribed_dim=2, output_dim=512)\n    elif encoder_type == \'hybrid\':\n        pixel_enc = ImpalaEncoder(\n            width=1,\n            stack_sizes=(16, cfg.model.henc, cfg.model.dstc),\n            num_blocks=2, dropout_rate=None, layer_norm=False,\n            input_channels=cfg.model.dobs, final_ln=True,\n            mlp_output_dim=512,\n            input_shape=(cfg.model.dobs, data_config.img_size, data_config.img_size),\n        )\n        return HybridEncoder(pixel_enc, prescribed_dim=2, prescribed_output_dim=128)\n    else:\n        raise ValueError(f"Unknown encoder_type: {encoder_type}")\n\n\n# ================================================================\n# Main training\n# ================================================================\n\ndef run_condition(mode, drive_base):\n    cond = CONDITIONS[mode]\n    encoder_type = cond.get(\'encoder_type\', \'free\')\n    locations_available = encoder_type in (\'prescribed\', \'hybrid\')\n\n    cfg = load_config("examples/ac_video_jepa/cfgs/train.yaml")\n    # Colab overrides\n    cfg.logging.log_wandb = False\n    cfg.data.num_workers = 0\n    cfg.data.pin_mem = False\n    cfg.data.persistent_workers = False\n    cfg.data.batch_size = 64\n    cfg.training.dtype = "float16"\n    cfg.model.compile = False\n    cfg.meta.load_model = False\n    cfg.meta.enable_plan_eval = False\n\n    # Ablation overrides\n    for key in (\'idm_coeff\', \'std_coeff\', \'cov_coeff\', \'sim_coeff_t\'):\n        if key in cond:\n            setattr(cfg.model.regularizer, key, cond[key])\n\n    loader, val_loader, data_config = init_data(\n        env_name=cfg.data.env_name, cfg_data=dict(cfg.data)\n    )\n    setup_device("auto")\n    setup_seed(cfg.meta.seed)\n    device = torch.device(DEVICE)\n\n    scaler = GradScaler(device.type, enabled=True)\n    local_folder = Path(f"/content/eb_jepa_runs/{mode}")\n    os.makedirs(local_folder, exist_ok=True)\n    steps_per_epoch = len(loader)\n    total_steps = cfg.optim.epochs * steps_per_epoch\n\n    # Build model\n    encoder = build_encoder(encoder_type, cfg, data_config)\n    mlp_dim = encoder.mlp_output_dim\n\n    # Get final_ln from encoder\n    enc_final_ln = getattr(encoder, \'final_ln\', None)\n    if enc_final_ln is None or isinstance(enc_final_ln, bool):\n        enc_final_ln = nn.LayerNorm(mlp_dim)\n\n    predictor = RNNPredictor(hidden_size=mlp_dim, final_ln=enc_final_ln)\n    aencoder = nn.Identity()\n\n    idm = InverseDynamicsModel(state_dim=mlp_dim, hidden_dim=256, action_dim=2).to(device)\n    regularizer = VC_IDM_Sim_Regularizer(\n        cov_coeff=cfg.model.regularizer.cov_coeff,\n        std_coeff=cfg.model.regularizer.std_coeff,\n        sim_coeff_t=cfg.model.regularizer.sim_coeff_t,\n        idm_coeff=cfg.model.regularizer.get("idm_coeff", 0.1),\n        idm=idm,\n        first_t_only=cfg.model.regularizer.get("first_t_only"),\n        spatial_as_samples=cfg.model.regularizer.spatial_as_samples,\n        idm_after_proj=cfg.model.regularizer.idm_after_proj,\n        sim_t_after_proj=cfg.model.regularizer.sim_t_after_proj,\n    )\n    ploss_fn = SquareLossSeq()\n\n    if locations_available:\n        jepa = PrescribedJEPA(encoder, aencoder, predictor, regularizer, ploss_fn).to(device)\n    else:\n        jepa = JEPA(encoder, aencoder, predictor, regularizer, ploss_fn).to(device)\n\n    xy_head = MLPXYHead(\n        input_shape=mlp_dim,\n        normalizer=loader.dataset.normalizer,\n    ).to(device)\n    xy_prober = JEPAProbe(jepa=jepa, head=xy_head, hcost=nn.MSELoss())\n\n    jepa_opt = AdamW(jepa.parameters(), lr=cfg.optim.lr,\n                     weight_decay=cfg.optim.get("weight_decay", 1e-6))\n    jepa_sched = CosineWithWarmup(jepa_opt, total_steps, warmup_ratio=0.1)\n    probe_opt = AdamW(xy_head.parameters(), lr=1e-3, weight_decay=1e-5)\n    probe_sched = CosineWithWarmup(probe_opt, total_steps, warmup_ratio=0.1)\n\n    start_epoch, start_step, results_log = _load_resume(\n        mode, drive_base, device, jepa, jepa_opt, jepa_sched, xy_head,\n        probe_opt, probe_sched\n    )\n\n    enc_params = sum(p.numel() for p in encoder.parameters())\n    pred_params = sum(p.numel() for p in predictor.parameters())\n    logger.info(f"=== {mode} === encoder={enc_params:,} predictor={pred_params:,}")\n    logger.info(f"Epochs: {start_epoch} -> {cfg.optim.epochs}, batches/epoch: {steps_per_epoch}")\n    logger.info(f"Config: cov={cfg.model.regularizer.cov_coeff} std={cfg.model.regularizer.std_coeff} "\n                f"sim={cfg.model.regularizer.sim_coeff_t} idm={cfg.model.regularizer.get(\'idm_coeff\', 0.1)}")\n\n    # ---- Training loop ----\n    SAVE_EVERY_N_BATCHES = 200  # mid-epoch checkpoint\n\n    for epoch in range(start_epoch, cfg.optim.epochs):\n        epoch_start = time.time()\n        ep_losses = {\'pred\': [], \'reg\': [], \'probe\': [], \'total\': []}\n        ep_reg = {\'std\': [], \'cov\': [], \'idm\': [], \'sim\': []}\n\n        # Determine which batch to start from (for mid-epoch resume)\n        skip_to = start_step if epoch == start_epoch else 0\n\n        pbar = tqdm(enumerate(loader), total=len(loader),\n                    desc=f"[{mode}] Ep {epoch}/{cfg.optim.epochs-1}")\n\n        for idx, (x, a, loc, _, _) in pbar:\n            # Skip batches already completed (mid-epoch resume)\n            if idx < skip_to:\n                if idx == 0:\n                    logger.info(f"Skipping {skip_to} batches (mid-epoch resume)...")\n                continue\n            x, a, loc = x.to(device), a.to(device), loc.to(device)\n\n            jepa_opt.zero_grad()\n            with autocast(device.type, enabled=True, dtype=torch.float16):\n                if locations_available:\n                    _, (jepa_loss, regl, _, regldict, pl) = jepa.unroll(\n                        x, a, nsteps=cfg.model.nsteps,\n                        unroll_mode="autoregressive", ctxt_window_time=1,\n                        compute_loss=True, locations=loc,\n                    )\n                else:\n                    _, (jepa_loss, regl, _, regldict, pl) = jepa.unroll(\n                        x, a, nsteps=cfg.model.nsteps,\n                        unroll_mode="autoregressive", ctxt_window_time=1,\n                        compute_loss=True,\n                    )\n\n            scaler.scale(jepa_loss).backward()\n            if cfg.optim.get("grad_clip_enc") and cfg.optim.get("grad_clip_pred"):\n                scaler.unscale_(jepa_opt)\n                torch.nn.utils.clip_grad_norm_(jepa.encoder.parameters(), cfg.optim.grad_clip_enc)\n                torch.nn.utils.clip_grad_norm_(jepa.predictor.parameters(), cfg.optim.grad_clip_pred)\n            scaler.step(jepa_opt)\n            scaler.update()\n            jepa_sched.step()\n\n            probe_opt.zero_grad()\n            with autocast(device.type, enabled=True, dtype=torch.float16):\n                with torch.no_grad():\n                    if locations_available:\n                        probe_state = jepa.encoder(x[:, :, :1], locations=loc[:, :, :1])\n                    else:\n                        probe_state = jepa.encoder(x[:, :, :1])\n                probe_pred = xy_head(probe_state.detach())\n                xy_loss = nn.functional.mse_loss(probe_pred, loc[:, :, :1])\n                xy_loss = loader.dataset.normalizer.unnormalize_mse(xy_loss)\n            scaler.scale(xy_loss).backward()\n            scaler.step(probe_opt)\n            scaler.update()\n            probe_sched.step()\n\n            ep_losses[\'pred\'].append(pl.item())\n            ep_losses[\'reg\'].append(regl.item())\n            ep_losses[\'probe\'].append(xy_loss.item())\n            ep_losses[\'total\'].append(jepa_loss.item())\n            ep_reg[\'std\'].append(regldict.get(\'std_loss\', 0))\n            ep_reg[\'cov\'].append(regldict.get(\'cov_loss\', 0))\n            ep_reg[\'idm\'].append(regldict.get(\'idm_loss\', 0))\n            ep_reg[\'sim\'].append(regldict.get(\'sim_loss_t\', 0))\n\n            pbar.set_postfix({\n                \'pred\': f"{pl.item():.4f}",\n                \'reg\': f"{regl.item():.4f}",\n                \'probe\': f"{xy_loss.item():.4f}",\n            })\n\n            # Mid-epoch checkpoint every N batches\n            if (idx + 1) % SAVE_EVERY_N_BATCHES == 0 and idx + 1 < len(loader):\n                save_checkpoint(\n                    local_folder / "latest.pth.tar",\n                    model=jepa, optimizer=jepa_opt, scheduler=jepa_sched,\n                    epoch=epoch, step=epoch * steps_per_epoch + idx,\n                    xy_head_state_dict=xy_head.state_dict(),\n                    probe_optimizer_state_dict=probe_opt.state_dict(),\n                    probe_scheduler_state_dict=probe_sched.state_dict(),\n                    batch_idx=idx,\n                )\n                dst = _drive_dir(mode, drive_base)\n                shutil.copy2(str(local_folder / "latest.pth.tar"),\n                            os.path.join(dst, "latest.pth.tar"))\n                logger.info(f"Mid-epoch save: ep {epoch} batch {idx+1}/{len(loader)}")\n\n        epoch_time = time.time() - epoch_start\n        avg = {k: float(np.mean(v)) for k, v in ep_losses.items()}\n        avg_reg = {k: float(np.mean(v)) for k, v in ep_reg.items()}\n\n        # Planning eval (only for free — prescribed needs GCAgent patch)\n        sr, md = -1.0, -1.0\n        try:\n            if not locations_available:\n                with open("examples/ac_video_jepa/cfgs/planning_mppi.yaml") as f:\n                    plan_cfg = yaml.safe_load(f)\n                with open("examples/ac_video_jepa/cfgs/eval.yaml") as f:\n                    eval_cfg = yaml.safe_load(f)\n                _, _, env_config = init_data(\n                    env_name=cfg.data.env_name, cfg_data=dict(eval_cfg.get("data", {}))\n                )\n                def env_creator():\n                    from eb_jepa.datasets.two_rooms.env import DotWall\n                    return DotWall(config=env_config, **eval_cfg.get("env", {}))\n                plan_cfg["logging"] = {"tqdm_silent": True}\n                ef = local_folder / f"plan_ep{epoch}"\n                os.makedirs(ef, exist_ok=True)\n                er = main_eval(plan_cfg=plan_cfg, model=jepa, env_creator=env_creator,\n                               eval_folder=ef, num_episodes=10, loader=val_loader, prober=xy_prober)\n                sr = er[\'success_rate\']\n                md = er[\'mean_state_dist\']\n            else:\n                logger.info(f"Planning eval skipped for {mode} (prescribed encoder)")\n        except Exception as e:\n            logger.warning(f"Planning eval failed: {e}")\n\n        enc_stats = compute_encoder_stats(jepa, loader, device, locations_available)\n        enc_stats[\'epoch\'] = epoch\n\n        logger.info(f"[{mode}] Ep {epoch}: pred={avg[\'pred\']:.4f} reg={avg[\'reg\']:.4f} "\n                    f"probe={avg[\'probe\']:.4f} SR={sr} time={epoch_time:.0f}s")\n\n        epoch_result = {\n            \'epoch\': epoch, \'mode\': mode,\n            \'pred_loss\': avg[\'pred\'], \'reg_loss\': avg[\'reg\'],\n            \'probe_loss\': avg[\'probe\'], \'total_loss\': avg[\'total\'],\n            \'std_loss\': avg_reg[\'std\'], \'cov_loss\': avg_reg[\'cov\'],\n            \'idm_loss\': avg_reg[\'idm\'], \'sim_loss\': avg_reg[\'sim\'],\n            \'success_rate\': sr, \'mean_dist\': md, \'epoch_time\': epoch_time,\n            \'encoder_params\': enc_params, \'predictor_params\': pred_params,\n        }\n        results_log.append(epoch_result)\n\n        save_checkpoint(\n            local_folder / "latest.pth.tar",\n            model=jepa, optimizer=jepa_opt, scheduler=jepa_sched,\n            epoch=epoch, step=(epoch + 1) * steps_per_epoch,\n            xy_head_state_dict=xy_head.state_dict(),\n            probe_optimizer_state_dict=probe_opt.state_dict(),\n            probe_scheduler_state_dict=probe_sched.state_dict(),\n            batch_idx=-1,\n        )\n        _save_to_drive(mode, drive_base, str(local_folder), results_log, enc_stats)\n        logger.info(f"Saved to Drive: {_drive_dir(mode, drive_base)}")\n\n    logger.info(f"=== {mode} COMPLETE === {len(results_log)} epochs")\n    return results_log\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", type=str, required=True, choices=list(CONDITIONS.keys()))\n    parser.add_argument("--drive_base", type=str, default="/content/drive/MyDrive/eb_jepa_v3")\n    args = parser.parse_args()\n    run_condition(args.mode, args.drive_base)\n'

with open(script_path, 'w') as f:
    f.write(SCRIPT)
print(f'Written: {script_path} ({os.path.getsize(script_path)} bytes)')


In [ ]:
# CELL 4: Run one condition
MODE = 'prescribed'

import json, os, torch
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
rp = os.path.join(DRIVE_BASE, MODE, 'results.json')
if os.path.exists(rp):
    with open(rp) as f:
        done = json.load(f)
    print(f'{MODE}: {len(done)} epochs done')
    if done:
        last = done[-1]
        print(f'  Last: ep {last["epoch"]}, pred={last["pred_loss"]:.4f}, SR={last["success_rate"]}')
else:
    print(f'{MODE}: starting fresh')

ckpt_path = os.path.join(DRIVE_BASE, MODE, 'latest.pth.tar')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    batch_idx = ckpt.get('batch_idx', -1)
    if batch_idx >= 0:
        print(f'  Mid-epoch checkpoint: epoch {ckpt["epoch"]}, batch {batch_idx}/1562')
    else:
        print(f'  Full epoch checkpoint: epoch {ckpt["epoch"]}')
    del ckpt

!cd /content/eb_jepa && python experiments/prescribed_axes/run_experiment_v3.py --mode {MODE} --drive_base {DRIVE_BASE}

In [ ]:
# CELL 5: Results summary
import json, os

DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
modes = ['free', 'prescribed', 'hybrid', 'prescribed_no_idm', 'prescribed_no_vicreg', 'prescribed_no_sim']

print('=' * 85)
print(f'{"Mode":<25} {"Ep":>3} {"Pred":>8} {"Reg":>8} {"Probe":>8} {"SR":>6} {"Time":>6}')
print('-' * 85)
for m in modes:
    rp = os.path.join(DRIVE_BASE, m, 'results.json')
    if os.path.exists(rp):
        with open(rp) as f:
            d = json.load(f)
        if d:
            l = d[-1]
            print(f'{m:<25} {len(d):>3} {l["pred_loss"]:>8.4f} {l["reg_loss"]:>8.4f} '
                  f'{l["probe_loss"]:>8.4f} {l["success_rate"]:>6} {l["epoch_time"]:>5.0f}s')
    else:
        print(f'{m:<25}  --')
print('=' * 85)

In [ ]:
# CELL 6: Download all results
from google.colab import files
import shutil, os

DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
collect = '/content/eb_jepa_v3_download'
if os.path.exists(collect):
    shutil.rmtree(collect)
os.makedirs(collect)

for mode in os.listdir(DRIVE_BASE):
    md_path = os.path.join(DRIVE_BASE, mode)
    if not os.path.isdir(md_path):
        continue
    dst = os.path.join(collect, mode)
    os.makedirs(dst, exist_ok=True)
    for fn in ['results.json', 'encoder_stats.json']:
        src = os.path.join(md_path, fn)
        if os.path.exists(src):
            shutil.copy2(src, dst)

shutil.make_archive('/content/eb_jepa_v3_results', 'zip', collect)
files.download('/content/eb_jepa_v3_results.zip')
print('Downloaded!')